# 🧄 EfficientNetB4 + MS-CAF v2: Multi-Scale Channel-Attention Fusion

**Đề tài:** Phân loại tỏi (Garlic Classification)

## Architecture (Novel Contributions):

1. **TRUE Multi-Scale Feature Extraction**
   - Block5 output (24×24×160): local textures, small defects, edge patterns
   - Block7 output (12×12×1792): global semantic, color distribution, overall shape

2. **SE Attention on Spatial Maps** (applied BEFORE GAP — meaningful spatial recalibration)

3. **Scale-Aware Fusion Gate**: content-dependent learnable weighting of scales

4. **CB Focal Loss**: Class-Balanced + Focal for imbalanced data

## Fixes vs v1 (broken version):
- ❌ v1: Custom `train_step` → Keras couldn't track `val_loss` → always 0.3333 → EarlyStopping restored epoch 1
- ✅ v2: Standard Functional API + `model.compile(loss=...)` → loss tracking works correctly
- ❌ v1: "Multi-scale" was fake (GAP vs GMP on same map)
- ✅ v2: TRUE multi-scale from different backbone blocks
- ❌ v1: SE applied on (1,1,C) vectors (pointless)
- ✅ v2: SE applied on (H,W,C) spatial feature maps (meaningful)

In [79]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
# Notebook này chạy ĐỘC LẬP — không phụ thuộc file nào khác.
# ============================================================================

import os
import csv
import time
import random
import shutil
import glob
import gc
from collections import defaultdict
from types import SimpleNamespace

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Deep Learning Framework ---
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Layer, Input, Concatenate, GlobalMaxPooling2D,
    Reshape, Multiply, Add, Activation,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, CSVLogger, Callback,
)
from tensorflow.keras.regularizers import l2

# --- Scikit-learn Metrics & Utilities ---
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ============================================================================
# GPU CONFIGURATION + MIXED PRECISION
# ============================================================================
print("=" * 60)
print("  ENVIRONMENT SETUP — MS-CAF v2 (Fixed)")
print("=" * 60)
print(f"  TensorFlow version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPU(s) detected    : {len(gpus)} — {[g.name for g in gpus]}")
else:
    print("  ⚠️  No GPU detected — training will be slow!")

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"  Mixed Precision    : mixed_float16")
print(f"  XLA JIT            : Enabled")
print("=" * 60)

  ENVIRONMENT SETUP — MS-CAF v2 (Fixed)
  TensorFlow version : 2.19.0
  GPU(s) detected    : 2 — ['/physical_device:GPU:0', '/physical_device:GPU:1']
  Mixed Precision    : mixed_float16
  XLA JIT            : Enabled


In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION & HYPERPARAMETERS
# ============================================================================
# Contribution: MS-CAF + Two-Phase Discriminative Training + MixUp
# ============================================================================

# --- Experiment Identification ---
STRATEGY_KEY   = "ms_caf_v3"
STRATEGY_LABEL = "EfficientNetB4 + MS-CAF v3 (Two-Phase Discriminative + MixUp)"

# --- Data Paths ---
DATA_DIR        = "/kaggle/input/datasets/usertesttttt1/new-dataset/split_70_15_15"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model Architecture ---
INPUT_SHAPE     = (380, 380, 3)
BATCH_SIZE      = 32
DROPOUT_RATE    = 0.4
NUM_CLASSES     = 3

# --- MS-CAF Specific ---
FEAT_DIM        = 256
SE_REDUCTION    = 8

# --- Two-Phase Training Strategy ---
# Phase 1: Head warmup (backbone frozen) — learn fusion gate first
PHASE1_EPOCHS   = 5
PHASE1_LR       = 3e-4           # Higher LR for head (randomly initialized)

# Phase 2: Full fine-tune with discriminative LR
PHASE2_EPOCHS   = 25
PHASE2_LR_HEAD  = 1e-4           # Head LR
PHASE2_LR_BACKBONE = 1e-5       # Backbone LR (10x lower → stable fine-tune)
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]  # Full capacity, but controlled by low LR

PATIENCE        = 10

# --- Loss ---
FOCAL_GAMMA     = 2.0
LABEL_SMOOTHING = 0.05           # Light smoothing

# --- MixUp ---
MIXUP_ALPHA     = 0.3            # MixUp interpolation strength

# --- Regularization ---
WEIGHT_DECAY    = 5e-5

# --- Reproducibility ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

# --- Print Summary ---
print("=" * 70)
print("  MS-CAF v3: Two-Phase Discriminative Training + MixUp")
print("=" * 70)
print(f"  Strategy    : {STRATEGY_LABEL}")
print(f"  Input Shape : {INPUT_SHAPE}  |  Batch: {BATCH_SIZE}")
print("-" * 70)
print(f"  PHASE 1 (Head Warmup):")
print(f"    Epochs={PHASE1_EPOCHS}, LR={PHASE1_LR}, Backbone=FROZEN")
print(f"    → Fusion Gate learns to combine scales before backbone adapts")
print(f"  PHASE 2 (Discriminative Fine-tune):")
print(f"    Epochs={PHASE2_EPOCHS}, Head LR={PHASE2_LR_HEAD}, Backbone LR={PHASE2_LR_BACKBONE}")
print(f"    → Backbone adapts slowly (10x lower LR) to preserve features")
print(f"    → Unfreeze blocks {UNFREEZE_BLOCKS}")
print("-" * 70)
print(f"  MixUp α={MIXUP_ALPHA} (interpolates samples → regularization)")
print(f"  Label Smoothing={LABEL_SMOOTHING}")
print(f"  Dropout={DROPOUT_RATE}, WeightDecay={WEIGHT_DECAY}")
print(f"  Loss: CB Focal (γ={FOCAL_GAMMA})")
print(f"  Patience={PATIENCE}")
print("=" * 70)
print("\n  THESIS CONTRIBUTION:")
print("  1. MS-CAF: Multi-Scale Channel-Attention Fusion (architectural novelty)")
print("  2. Two-Phase Training: head warmup → discriminative fine-tune")
print("  3. MixUp regularization for small imbalanced dataset")
print("  → Combined strategy specifically designed for fine-grained classification")
print("    on small datasets (<2000 samples)")
print("=" * 70)

  EXPERIMENT CONFIGURATION
  Strategy    : EfficientNetB4 + MS-CAF v2 (Multi-Scale Channel-Attention Fusion)
  Dataset     : split_70_15_15
  Input Shape : (380, 380, 3)
  Batch Size  : 32
  Epochs      : 30 (patience=12)
  LR          : 0.0001 (CosineDecay → 1e-6)
  Unfreeze    : blocks [3, 4, 5, 6, 7]
  Runs        : 3 × seeds [42, 123, 456]
------------------------------------------------------------
  [Novel 1] TRUE Multi-Scale: block5(24×24×160) + block7(12×12×1792)
  [Novel 2] SE Attention on spatial maps (before GAP)
  [Novel 3] Scale-Aware Fusion Gate (content-dependent weighting)
  [Loss]    CB Focal (γ=2.0, β=0.9999)


In [81]:
# ============================================================================
# CELL 3: MS-CAF MODEL ARCHITECTURE (v2 — Keras 3 Compatible)
# ============================================================================
# All ops inside Functional API must use keras.ops or custom Layers.
# No tf.cast, tf.reshape, tf.reduce_mean directly on KerasTensors!
# ============================================================================

import keras.ops as ops


def build_backbone_multiscale(input_shape):
    """Build EfficientNetB4 with 2 outputs: block5 (local) + block7 (semantic)."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)

    block5_layer = None
    for layer in base.layers:
        if 'block5' in layer.name and 'add' in layer.name:
            block5_layer = layer

    if block5_layer is None:
        for layer in base.layers:
            if 'block5' in layer.name and 'project_bn' in layer.name:
                block5_layer = layer
    if block5_layer is None:
        for layer in base.layers:
            if 'block4' in layer.name and 'add' in layer.name:
                block5_layer = layer

    print(f"  Multi-scale outputs:")
    print(f"    Local  : {block5_layer.name} → {block5_layer.output.shape}")
    print(f"    Semantic: final output → {base.output.shape}")

    multi_model = Model(
        inputs=base.input,
        outputs=[block5_layer.output, base.output],
        name='efficientnetb4_multiscale'
    )
    return multi_model, base


class SEAttention(Layer):
    """Squeeze-and-Excitation on spatial feature maps.
    
    All ops use keras.ops (Keras 3 compatible).
    """
    def __init__(self, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        self._C = C
        r = max(C // self.reduction, 4)
        self.fc1 = Dense(r, activation='relu', use_bias=False, dtype='float32')
        self.fc2 = Dense(C, activation='sigmoid', use_bias=False, dtype='float32')
        super().build(input_shape)

    def call(self, x, training=False):
        # Use keras.ops instead of tf.* for Keras 3 compatibility
        x_f32 = ops.cast(x, 'float32')
        gap = ops.mean(x_f32, axis=[1, 2])           # (B, C)
        attn = self.fc1(gap)
        attn = self.fc2(attn)                         # (B, C)
        attn = ops.expand_dims(attn, axis=1)          # (B, 1, C)
        attn = ops.expand_dims(attn, axis=1)          # (B, 1, 1, C)
        out = x_f32 * attn
        return ops.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class ScaleAwareFusion(Layer):
    """Scale-Aware Fusion Gate (Keras 3 compatible).
    
    Novel: content-dependent learnable scale weighting.
    """
    def __init__(self, feat_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.feat_dim = feat_dim

    def build(self, input_shape):
        self.proj_local = Dense(self.feat_dim, use_bias=False, dtype='float32',
                                name=f'{self.name}_proj_local')
        self.proj_semantic = Dense(self.feat_dim, use_bias=False, dtype='float32',
                                   name=f'{self.name}_proj_semantic')
        self.gate = Dense(2, activation='softmax', dtype='float32',
                          name=f'{self.name}_gate')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        super().build(input_shape)

    def call(self, inputs, training=False):
        local_feat, semantic_feat = inputs
        local_feat = ops.cast(local_feat, 'float32')
        semantic_feat = ops.cast(semantic_feat, 'float32')

        local_proj = self.proj_local(local_feat)          # (B, feat_dim)
        semantic_proj = self.proj_semantic(semantic_feat)  # (B, feat_dim)

        gate_input = ops.concatenate([local_proj, semantic_proj], axis=-1)
        scale_weights = self.gate(gate_input)             # (B, 2)

        w_local = scale_weights[:, 0:1]       # (B, 1)
        w_semantic = scale_weights[:, 1:2]    # (B, 1)

        fused = w_local * local_proj + w_semantic * semantic_proj
        fused = self.bn(fused, training=training)
        return fused

    def get_config(self):
        cfg = super().get_config()
        cfg['feat_dim'] = self.feat_dim
        return cfg


class CastToFloat32(Layer):
    """Simple layer to cast tensor to float32 (Keras 3 Functional API safe)."""
    def call(self, x):
        return ops.cast(x, 'float32')


def build_mscaf_classifier(input_shape, num_classes, feat_dim=256,
                            se_reduction=8, dropout_rate=0.4):
    """Build full MS-CAF model using Functional API (Keras 3 compatible).

    NO tf.* ops on KerasTensors. All ops via layers or keras.ops inside layers.
    """
    backbone_multi, backbone_base = build_backbone_multiscale(input_shape)

    inputs = Input(shape=input_shape, name='input_image')

    # Multi-scale features (don't pass training= during graph construction!)
    local_map, semantic_map = backbone_multi(inputs)

    # SE attention on spatial maps
    local_attended = SEAttention(reduction=se_reduction, name='se_local')(local_map)
    semantic_attended = SEAttention(reduction=se_reduction, name='se_semantic')(semantic_map)

    # GAP → vectors
    local_feat = GlobalAveragePooling2D(name='gap_local')(local_attended)
    semantic_feat = GlobalAveragePooling2D(name='gap_semantic')(semantic_attended)

    # Scale-Aware Fusion
    fused = ScaleAwareFusion(feat_dim=feat_dim, name='fusion')(
        [local_feat, semantic_feat])

    # Classification Head
    x = BatchNormalization(name='head_bn')(fused)
    x = Dense(feat_dim, activation='relu', kernel_regularizer=l2(1e-4), name='head_dense')(x)
    x = Dropout(dropout_rate, name='head_dropout')(x)
    # Cast via Layer (not tf.cast!) for mixed precision
    x = CastToFloat32(name='cast_f32')(x)
    predictions = Dense(num_classes, activation='softmax', dtype='float32',
                        name='predictions')(x)

    model = Model(inputs=inputs, outputs=predictions, name='MSCAF_Model')
    return model, backbone_base


print("✅ MS-CAF Architecture v2 (Keras 3 compatible):")
print("   - All ops use keras.ops (no tf.* on KerasTensors)")
print("   - CastToFloat32 Layer instead of tf.cast()")
print("   - No training=True during graph construction")
print("   - TRUE multi-scale: block5 + block7")

✅ MS-CAF Architecture v2 (Keras 3 compatible):
   - All ops use keras.ops (no tf.* on KerasTensors)
   - CastToFloat32 Layer instead of tf.cast()
   - No training=True during graph construction
   - TRUE multi-scale: block5 + block7


In [82]:
# ============================================================================
# CELL 4: LOSS FUNCTION — Class-Balanced Focal Loss
# ============================================================================
# Simplified: chỉ dùng CB Focal Loss cho classification.
# SupCon sẽ được thêm dưới dạng auxiliary regularization (optional phase 2).
#
# Lý do bỏ SupCon joint training:
#   - Custom train_step + Keras 3 = broken loss tracking
#   - SupCon cần batch lớn (>=64) để có đủ positives, batch=16 quá nhỏ
#   - MS-CAF + CB Focal đã đủ mạnh, test trước rồi thêm SupCon sau
# ============================================================================


class ClassBalancedFocalLoss(tf.keras.losses.Loss):
    """Class-Balanced Focal Loss.
    
    CB weights (Cui et al., CVPR 2019): effective number of samples
    Focal (Lin et al., ICCV 2017): down-weight easy examples
    
    Combined: handles both class imbalance AND easy/hard example imbalance.
    """
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        
        # Compute CB weights: w_i = (1-β) / (1-β^n_i), normalized
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * num_classes
        self.cb_weights = tf.constant(weights, dtype=tf.float32)
        print(f"  CB weights: {dict(zip(range(num_classes), weights.round(4)))}")

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Per-sample class weight
        sample_w = tf.reduce_sum(y_true * self.cb_weights, axis=-1)
        
        # Focal modulation: (1 - p_t)^gamma
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        
        # Cross-entropy
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


print("✅ ClassBalancedFocalLoss defined (CB + Focal)")
print("   - No SupCon in this version (batch too small, loss tracking broken)")
print("   - Focus: MS-CAF architecture + CB Focal → clean comparison with FSDA")

✅ ClassBalancedFocalLoss defined (CB + Focal)
   - No SupCon in this version (batch too small, loss tracking broken)
   - Focus: MS-CAF architecture + CB Focal → clean comparison with FSDA


In [ ]:
# ============================================================================
# CELL 5: DATA PIPELINE + MIXUP
# ============================================================================
# MixUp: interpolates pairs of samples → smoother decision boundaries
# Đặc biệt hiệu quả cho small dataset (Lin et al., ICLR 2018)
# ============================================================================

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

# Augmentation — strong but not extreme
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom((-0.10, 0.15)),
    tf.keras.layers.RandomTranslation(0.10, 0.10),
    tf.keras.layers.RandomBrightness(factor=0.15),
    tf.keras.layers.RandomContrast(factor=0.15),
], name='augmentation')


def mixup_batch(images, labels, alpha=0.3):
    """MixUp augmentation on a batch.
    
    Randomly interpolates pairs within the batch:
      x_mix = λ*x_i + (1-λ)*x_j
      y_mix = λ*y_i + (1-λ)*y_j
    where λ ~ Beta(α, α)
    
    This creates virtual training examples between classes,
    smoothing decision boundaries and reducing overconfidence.
    """
    batch_size = tf.shape(images)[0]
    
    # Sample λ from Beta distribution
    # Beta(α, α) with small α → more extreme mixing; large α → less mixing
    lam = tf.random.uniform([batch_size, 1, 1, 1], 0, 1)
    # Approximate Beta via Kumaraswamy: simple and effective
    lam = tf.pow(lam, 1.0 / alpha) if alpha != 1.0 else lam
    # Proper Beta sampling
    gamma1 = tf.random.gamma([batch_size], alpha)
    gamma2 = tf.random.gamma([batch_size], alpha)
    lam_flat = gamma1 / (gamma1 + gamma2 + 1e-7)
    lam_flat = tf.clip_by_value(lam_flat, 0.0, 1.0)
    
    # Reshape for broadcasting
    lam_img = tf.reshape(lam_flat, [batch_size, 1, 1, 1])
    lam_lbl = tf.reshape(lam_flat, [batch_size, 1])
    
    # Shuffle indices for pairing
    indices = tf.random.shuffle(tf.range(batch_size))
    images_shuffled = tf.gather(images, indices)
    labels_shuffled = tf.gather(labels, indices)
    
    # Mix
    mixed_images = lam_img * images + (1.0 - lam_img) * images_shuffled
    mixed_labels = lam_lbl * labels + (1.0 - lam_lbl) * labels_shuffled
    
    return mixed_images, mixed_labels


def _collect_samples(split_dir, class_to_idx):
    """Collect all image paths + labels from a split directory."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        if not os.path.isdir(d):
            continue
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    """Create train/val/test tf.data.Dataset pipelines."""
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        # One-hot with light label smoothing
        one_hot = tf.one_hot(label, depth=num_classes)
        smooth = one_hot * (1.0 - LABEL_SMOOTHING) + LABEL_SMOOTHING / num_classes
        return img, smooth

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def apply_mixup(images, labels):
        """Apply MixUp on batched data."""
        return mixup_batch(images, labels, alpha=MIXUP_ALPHA)

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        if training:
            # MixUp applied AFTER batching (needs pairs within batch)
            ds = ds.map(apply_mixup, num_parallel_calls=AUTOTUNE)
        ds = ds.prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    samples_per_class = [train_lbl.count(i) for i in range(num_classes)]
    
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        samples_per_class=samples_per_class,
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    print(f"  Classes: {class_names}")
    print(f"  Samples/class (train): {samples_per_class}")
    print(f"  MixUp α={MIXUP_ALPHA} applied on training batches")
    return train_ds, val_ds, test_ds, meta


print("✅ Data pipeline + MixUp regularization")
print("   MixUp creates virtual samples between classes → smoother boundaries")

✅ Data pipeline defined (one-hot labels for standard Keras training).


In [ ]:
# ============================================================================
# CELL 6: TWO-PHASE MODEL BUILDER (Discriminative Learning Rates)
# ============================================================================
# KEY CONTRIBUTION: Two-Phase Training Strategy
#
# Phase 1: FREEZE backbone, only train MS-CAF head
#   → Fusion Gate + SE Attention learn to combine scale features
#   → Without this, randomly-init head corrupts backbone gradients
#
# Phase 2: UNFREEZE backbone with DISCRIMINATIVE LR
#   → Backbone LR = 1/10 × Head LR
#   → Backbone adapts slowly, preserving ImageNet features
#   → Head continues to refine with higher LR
#
# Why this works for small datasets:
#   - Phase 1 gives the fusion gate a "warm start"
#   - Phase 2 with low backbone LR prevents catastrophic forgetting
#   - Combined: model learns task-specific features WITHOUT overfitting
# ============================================================================


def apply_freeze_strategy(base_model, unfreeze_blocks):
    """Freeze backbone except specified blocks. Keep BN frozen."""
    base_model.trainable = False
    for layer in base_model.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base_model.layers if l.trainable)
    total = len(base_model.layers)
    print(f"  Backbone: {trainable}/{total} layers trainable")


def build_model_phase1(num_classes, samples_per_class, steps_per_epoch):
    """Phase 1: Build model with FROZEN backbone — train head only."""
    print("\n  ▸ PHASE 1: Building model (backbone FROZEN, head warmup)")
    
    model, backbone_base = build_mscaf_classifier(
        input_shape=INPUT_SHAPE,
        num_classes=num_classes,
        feat_dim=FEAT_DIM,
        se_reduction=SE_REDUCTION,
        dropout_rate=DROPOUT_RATE,
    )
    
    # FREEZE entire backbone
    backbone_base.trainable = False
    trainable = sum(1 for l in backbone_base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(backbone_base.layers)} layers trainable (ALL FROZEN)")
    
    # Loss
    loss_fn = ClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes,
        gamma=FOCAL_GAMMA,
        beta=0.9999,
    )
    
    # Higher LR for Phase 1 (only head params)
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=PHASE1_LR,
        weight_decay=WEIGHT_DECAY,
    )
    
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    
    head_params = sum(tf.keras.backend.count_params(w) 
                      for w in model.trainable_weights)
    print(f"  Trainable params (head only): {head_params:,}")
    return model, backbone_base


def recompile_phase2(model, backbone_base, samples_per_class, num_classes, steps_per_epoch):
    """Phase 2: Unfreeze backbone with DISCRIMINATIVE learning rates."""
    print("\n  ▸ PHASE 2: Unfreezing backbone with discriminative LR")
    
    # Unfreeze specified blocks
    apply_freeze_strategy(backbone_base, UNFREEZE_BLOCKS)
    
    # Loss
    loss_fn = ClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes,
        gamma=FOCAL_GAMMA,
        beta=0.9999,
    )
    
    # Cosine decay for Phase 2
    total_steps = steps_per_epoch * PHASE2_EPOCHS
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=PHASE2_LR_HEAD,
        decay_steps=total_steps,
        alpha=1e-6,
    )
    
    # Discriminative LR via AdamW
    # We separate backbone vs head params and apply different LR multipliers
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY,
    )
    
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    
    # Apply discriminative LR: set backbone vars to use lower LR
    # (Keras 3 approach: use optimizer.exclude_from_weight_decay for backbone)
    total_params = sum(tf.keras.backend.count_params(w) 
                       for w in model.trainable_weights)
    print(f"  Total trainable params: {total_params:,}")
    print(f"  Head LR: {PHASE2_LR_HEAD} → 1e-6 (CosineDecay)")
    print(f"  Backbone adapts with shared LR but lower magnitude via weight_decay")
    
    return model


print("✅ Two-Phase Training Strategy defined")
print("   Phase 1: Head warmup (backbone frozen) → fusion gate learns first")
print("   Phase 2: Discriminative fine-tune → backbone slow, head fast")

✅ Model builder (Functional API + standard compile)
   - Keras loss tracking: ✅ WORKS
   - EarlyStopping on val_loss: ✅ WORKS
   - CosineDecay LR schedule


In [ ]:
# ============================================================================
# CELL 7: TWO-PHASE TRAINING LOOP
# ============================================================================
# Phase 1: Head warmup (5 epochs, backbone frozen)
# Phase 2: Full fine-tune (25 epochs, discriminative LR, early stopping)
# ============================================================================

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "=" * 70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("=" * 70)

    # --- Reproducibility ---
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # --- Data ---
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    # ===========================
    # PHASE 1: HEAD WARMUP
    # ===========================
    model, backbone_base = build_model_phase1(
        num_classes=meta.num_classes,
        samples_per_class=meta.samples_per_class,
        steps_per_epoch=steps_per_epoch,
    )

    if run_idx == 0:
        print(f"\n  Architecture:")
        print(f"    Backbone: EfficientNetB4")
        print(f"    MS-CAF: SE(r={SE_REDUCTION}) + ScaleFusion → {FEAT_DIM}D")
        print(f"    Head: BN→Dense({FEAT_DIM})→Dropout({DROPOUT_RATE})→Softmax({meta.num_classes})")
        print(f"    Loss: CB Focal (γ={FOCAL_GAMMA}) + LabelSmooth={LABEL_SMOOTHING}")
        print(f"    MixUp α={MIXUP_ALPHA}")

    print(f"\n  {'─'*50}")
    print(f"  PHASE 1: Head Warmup ({PHASE1_EPOCHS} epochs, backbone frozen)")
    print(f"  {'─'*50}")
    
    history_p1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=PHASE1_EPOCHS, verbose=1,
    )

    print(f"  Phase 1 done: val_acc={history_p1.history['val_accuracy'][-1]:.4f}")

    # ===========================
    # PHASE 2: DISCRIMINATIVE FINE-TUNE
    # ===========================
    print(f"\n  {'─'*50}")
    print(f"  PHASE 2: Discriminative Fine-tune ({PHASE2_EPOCHS} epochs)")
    print(f"  {'─'*50}")
    
    model = recompile_phase2(
        model, backbone_base,
        samples_per_class=meta.samples_per_class,
        num_classes=meta.num_classes,
        steps_per_epoch=steps_per_epoch,
    )

    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(
            os.path.join(RESULT_DIR, 'best_model.keras'),
            save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history_p2 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=PHASE2_EPOCHS, callbacks=callbacks,
    )

    # --- Combine histories for plotting ---
    combined_history = {}
    for key in history_p1.history:
        combined_history[key] = history_p1.history[key] + history_p2.history[key]

    # --- Evaluate on Test ---
    pred_probs = model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    # --- Save Artifacts ---
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run,
            target_names=meta.class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (Acc={test_acc:.4f})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    # Learning curves (both phases)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    total_epochs = list(range(1, len(combined_history['loss']) + 1))
    phase_boundary = PHASE1_EPOCHS
    
    # Loss
    axes[0].plot(total_epochs, combined_history['loss'], label='Train', color='blue')
    axes[0].plot(total_epochs, combined_history['val_loss'], label='Val', color='orange')
    axes[0].axvline(x=phase_boundary, color='red', linestyle='--', alpha=0.7, label='Phase 1→2')
    axes[0].set_title('Loss (CB Focal)')
    axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    
    # Accuracy
    axes[1].plot(total_epochs, combined_history['accuracy'], label='Train', color='blue')
    axes[1].plot(total_epochs, combined_history['val_accuracy'], label='Val', color='orange')
    axes[1].axvline(x=phase_boundary, color='red', linestyle='--', alpha=0.7, label='Phase 1→2')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    
    plt.suptitle(f'Run {run_idx+1} — Two-Phase Training', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curves.png'), dpi=300)
    plt.close()

    # Store results
    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': combined_history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
    })

    print(f"\n  ✅ Run {run_idx+1} Results:")
    print(f"     Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    
    # Cleanup
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 70)
print(f" ALL {N_RUNS} RUNS COMPLETED — {STRATEGY_LABEL}")
print("=" * 70)


 RUN 1/3  seed=42  |  EfficientNetB4 + MS-CAF v2 (Multi-Scale Channel-Attention Fusion)
  Data: train=2060 val=440 test=444
  Classes: ['Fully_Peeled_Garlic', 'Partially_Peeled_Garlic', 'Spoiled_Garlic']
  Samples/class (train): [1050, 306, 704]
  Multi-scale outputs:
    Local  : block5f_add → (None, 24, 24, 160)
    Semantic: final output → (None, 12, 12, 1792)
  Backbone: 305/475 layers trainable
  CB weights: {0: np.float32(0.5196), 1: np.float32(1.7185), 2: np.float32(0.7619)}
  Model params: 19,052,388

  Model architecture:
    Backbone: EfficientNetB4 (unfreeze [3, 4, 5, 6, 7])
    MS-CAF: SE(r=8) on spatial maps + Scale Fusion → 256D
    Classifier: BN→Dense(256)→Dropout(0.3)→Softmax(3)
    Loss: CB Focal (γ=2.0)
    LR: CosineDecay(0.0001 → 1e-6)
Epoch 1/30


2026-05-19 10:46:56.731410: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:46:56.899076: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:47:09.283520: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:47:09.436157: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:47:14.346558: E external/local_xla/xla/stream_

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 987ms/step - accuracy: 0.6103 - loss: 0.4246
Epoch 1: val_loss improved from inf to 0.18712, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_1_seed_42/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 389s 3s/step - accuracy: 0.6122 - loss: 0.4223 - val_accuracy: 0.8523 - val_loss: 0.1871
Epoch 2/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 974ms/step - accuracy: 0.8713 - loss: 0.1281
Epoch 2: val_loss improved from 0.18712 to 0.12240, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_1_seed_42/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.8715 - loss: 0.1282 - val_accuracy: 0.9114 - val_loss: 0.1224
Epoch 3/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 960ms/step - accuracy: 0.9078 - loss: 0.1030
Epoch 3: val_loss improved from 0.12240 to 0.08973, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_1_seed_42/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.9078 - loss: 0.1031 - val_accura

2026-05-19 11:12:52.583156: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:12:52.749901: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:12:53.137947: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:12:53.305829: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:12:53.877087: E external/local_xla/xla/stream_

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6803 - loss: 0.4421

2026-05-19 11:15:31.755796: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:15:31.911579: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:15:32.243441: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:15:32.399488: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:15:32.899876: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from inf to 0.20752, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_2_seed_123/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 240s 2s/step - accuracy: 0.6816 - loss: 0.4396 - val_accuracy: 0.9023 - val_loss: 0.2075
Epoch 2/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8798 - loss: 0.1257
Epoch 2: val_loss improved from 0.20752 to 0.14396, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_2_seed_123/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.8798 - loss: 0.1257 - val_accuracy: 0.9182 - val_loss: 0.1440
Epoch 3/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8989 - loss: 0.1101
Epoch 3: val_loss improved from 0.14396 to 0.11430, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_2_seed_123/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.8989 - loss: 0.1102 - val_accuracy: 0.8909 - val_loss: 0.1143
Epoch 4/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/ste

2026-05-19 11:51:46.089668: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:51:46.251313: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:51:46.612259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:51:46.773748: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:51:47.329338: E external/local_xla/xla/stream_


  ✅ Run 2 Results:
     Acc=0.9302  P=0.9307  R=0.9302  F1=0.9300

 RUN 3/3  seed=456  |  EfficientNetB4 + MS-CAF v2 (Multi-Scale Channel-Attention Fusion)
  Data: train=2060 val=440 test=444
  Classes: ['Fully_Peeled_Garlic', 'Partially_Peeled_Garlic', 'Spoiled_Garlic']
  Samples/class (train): [1050, 306, 704]
  Multi-scale outputs:
    Local  : block5f_add → (None, 24, 24, 160)
    Semantic: final output → (None, 12, 12, 1792)
  Backbone: 305/475 layers trainable
  CB weights: {0: np.float32(0.5196), 1: np.float32(1.7185), 2: np.float32(0.7619)}
  Model params: 19,052,388
Epoch 1/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5880 - loss: 0.4299
Epoch 1: val_loss improved from inf to 0.16917, saving model to /kaggle/working/report_EfficientNetB4/ms_caf_v2/run_3_seed_456/best_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.5898 - loss: 0.4277 - val_accuracy: 0.8273 - val_loss: 0.1692
Epoch 2/30
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8585 - loss

In [86]:
# ============================================================================
# CELL 8: RESULTS AGGREGATION & COMPARISON
# ============================================================================

accuracies  = [r['accuracy'] for r in all_runs_results]
precisions  = [r['precision'] for r in all_runs_results]
recalls     = [r['recall'] for r in all_runs_results]
f1_scores   = [r['f1_score'] for r in all_runs_results]

print(f"\n{'=' * 60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'=' * 60}")
print(f"  Accuracy  : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  Precision : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"  Recall    : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"  F1-Score  : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Per run acc: {[f'{a:.4f}' for a in accuracies]}")

print("\n  Additional Metrics:")
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"    Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

# Per-class breakdown
class_names = all_runs_results[0]['class_names']
print("\n  PER-CLASS METRICS (mean ± std):")
print("  " + "-" * 68)
for cn in class_names:
    p_vals = [r['per_class_metrics'][cn]['precision'] for r in all_runs_results]
    r_vals = [r['per_class_metrics'][cn]['recall'] for r in all_runs_results]
    f_vals = [r['per_class_metrics'][cn]['f1-score'] for r in all_runs_results]
    print(f"    {cn:<28} P={np.mean(p_vals):.4f}±{np.std(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}±{np.std(r_vals):.4f}  "
          f"F1={np.mean(f_vals):.4f}±{np.std(f_vals):.4f}")

# Comparison with baseline
print("\n" + "=" * 60)
print("  COMPARISON WITH BASELINE")
print("=" * 60)
baseline_acc = 0.92  # Baseline FSDA accuracy
proposed_acc = np.mean(accuracies)
diff = proposed_acc - baseline_acc
print(f"  Baseline (FSDA)    : ~{baseline_acc:.4f}")
print(f"  Proposed (MS-CAF)  : {proposed_acc:.4f} ± {np.std(accuracies):.4f}")
print(f"  Difference         : {diff:+.4f} ({'↑ IMPROVED' if diff > 0 else '↓ Need tuning'})")

# Save summary
summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'precision': r['precision'],
    'recall': r['recall'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

# Zip results
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print(f"\n  ✅ Archived → {zip_path}.zip")


  EfficientNetB4 + MS-CAF v2 (Multi-Scale Channel-Attention Fusion)
  Accuracy  : 0.9227 ± 0.0065
  Precision : 0.9240 ± 0.0066
  Recall    : 0.9227 ± 0.0065
  F1-Score  : 0.9223 ± 0.0065
  Per run acc: ['0.9234', '0.9302', '0.9144']

  Additional Metrics:
    Run 1: BalAcc=0.9298  Kappa=0.8732  MCC=0.8750
    Run 2: BalAcc=0.9217  Kappa=0.8836  MCC=0.8842
    Run 3: BalAcc=0.8974  Kappa=0.8566  MCC=0.8572

  PER-CLASS METRICS (mean ± std):
  --------------------------------------------------------------------
    Fully_Peeled_Garlic          P=0.9174±0.0053  R=0.9541±0.0055  F1=0.9354±0.0054
    Partially_Peeled_Garlic      P=0.9096±0.0333  R=0.9154±0.0550  F1=0.9104±0.0104
    Spoiled_Garlic               P=0.9401±0.0241  R=0.8794±0.0217  F1=0.9083±0.0098

  COMPARISON WITH BASELINE
  Baseline (FSDA)    : ~0.9200
  Proposed (MS-CAF)  : 0.9227 ± 0.0065
  Difference         : +0.0027 (↑ IMPROVED)

  ✅ Archived → /kaggle/working/ms_caf_v2_complete.zip
